# SARIMA – Daily Waste Forecast per Canteen Section

Trains a separate SARIMA (SARIMAX) model per canteen section using daily aggregated waste data.
Features: `Foot_Traffic`, `is_holiday`, `is_special_day` as exogenous regressors.
Split: chronological train / validation / test (no leakage).
Hyperparameter tuning via expanding-window cross-validation on the training set only.
Goal: given a date, predict the upcoming 7-day waste per section.

**Improvements applied:**
- Log‑transform target (log1p) to stabilise variance and improve spike learning.
- Grid search over (p,d,q) and seasonal (P,D,Q,7) orders.
- Cross-validation with 7‑day forecast horizon.
- Future regressor values estimated from rolling 7‑day mean of foot traffic.
- Handles multiple canteen sections separately.

**Note:** SARIMAX from `statsmodels` is used to incorporate exogenous regressors.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


## Configuration

In [3]:
DATA_PATH  = 'data/food_waste_features.csv'
MODEL_DIR  = 'deployment_models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Chronological split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Hyperparameter grid for SARIMAX (p,d,q)(P,D,Q,7)
# Note: d and D are fixed by stationarity tests; here we let them vary.
# Use small ranges to keep grid manageable.
P_GRID = [0, 1]      # AR order
D_GRID = [0, 1]      # Differencing order
Q_GRID = [0, 1]      # MA order
P_SEASONAL_GRID = [0, 1]
D_SEASONAL_GRID = [0, 1]
Q_SEASONAL_GRID = [0, 1]
SEASONAL_PERIOD = 7   # weekly seasonality (daily data)

FORECAST_HORIZON = 7   # days to forecast

print('Config ready.')

Config ready.


## Load & Aggregate Data

The raw data has 30‑min meal‑slot timestamps (~3 meals × 2 slots/meal = 6 records/day/section).
Sub‑daily resampling yields >97% empty bins, so we aggregate to **daily** totals per section.
Regressors (`Foot_Traffic`, `is_holiday`, `is_special_day`) are summed/max‑aggregated per day.

In [4]:
df = pd.read_csv(DATA_PATH, parse_dates=['time_bin'])
print('Raw shape:', df.shape)
df.head()

Raw shape: (99700, 7)


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,is_holiday,is_special_day
0,A,2015-01-01 07:00:00,0.090,0.14,1.20,1,1
1,A,2015-01-01 08:00:00,0.007,0.06,0.07,1,1
2,A,2015-01-01 08:30:00,0.026,0.05,0.07,1,1
3,A,2015-01-01 11:30:00,0.074,0.22,0.07,1,1
4,A,2015-01-01 13:30:00,0.036,0.11,0.07,1,1


In [5]:
df['date'] = df['time_bin'].dt.normalize()  # floor to day

daily = (
    df.groupby(['Canteen_Section', 'date'])
      .agg(
          y            = ('Waste_Weight_kg', 'sum'),
          foot_traffic = ('Foot_Traffic',    'sum'),
          is_holiday   = ('is_holiday',      'max'),
          is_special_day = ('is_special_day','max'),
      )
      .reset_index()
      .rename(columns={'date': 'ds', 'Canteen_Section': 'section'})
)

# Log‑transform target to stabilise variance
daily['y'] = np.log1p(daily['y'])

sections = sorted(daily['section'].unique())
print('Daily shape:', daily.shape)
print('Sections:', sections)
daily.head()

Daily shape: (15487, 6)
Sections: ['A', 'B', 'C', 'D']


,section,ds,y,foot_traffic,is_holiday,is_special_day
0,A,2015-01-01,0.339325,1.62,1,1
1,A,2015-01-02,0.878797,28.99,0,0
2,A,2015-01-03,0.692647,9.90,0,1
3,A,2015-01-04,0.227136,8.39,0,1
4,A,2015-01-05,0.398776,23.16,0,0


## Chronological Train / Validation / Test Split

Splits are based on the **global** date range so all sections share the same cutoffs.
This prevents any future data leaking into training.

In [6]:
all_dates = sorted(daily['ds'].unique())
n = len(all_dates)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_end = all_dates[n_train - 1]
val_end   = all_dates[n_train + n_val - 1]
test_start = all_dates[n_train + n_val]

print(f'Total unique days : {n}')
print(f'Train  : up to      {train_end.date()}  ({n_train} days)')
print(f'Val    : {all_dates[n_train].date()} – {val_end.date()}  ({n_val} days)')
print(f'Test   : {test_start.date()} – {all_dates[-1].date()}  ({n - n_train - n_val} days)')

Total unique days : 3875
Train  : up to      2022-06-04  (2712 days)
Val    : 2022-06-05 – 2024-01-06  (581 days)
Test   : 2024-01-07 – 2025-08-10  (582 days)


## Helper Functions

In [7]:
def split_section(df_sec):
    """Return train/val/test DataFrames for one section (no leakage)."""
    train = df_sec[df_sec['ds'] <= train_end].copy()
    val   = df_sec[(df_sec['ds'] > train_end) & (df_sec['ds'] <= val_end)].copy()
    test  = df_sec[df_sec['ds'] > val_end].copy()
    return train, val, test

def metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() else np.nan
    r2   = r2_score(y_true, y_pred)
    return dict(RMSE=rmse, MAE=mae, MAPE=mape, R2=r2)

def expand_window_cv(y, exog, param_grid, horizon=7, n_splits=5):
    """
    Perform expanding window cross-validation for SARIMAX.
    Returns average RMSE over validation folds.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=horizon)
    rmse_list = []
    for train_idx, test_idx in tscv.split(y):
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        if exog is not None:
            exog_train = exog.iloc[train_idx]
            exog_test  = exog.iloc[test_idx]
        else:
            exog_train = exog_test = None

        try:
            model = SARIMAX(y_train, exog=exog_train,
                            order=param_grid['order'],
                            seasonal_order=param_grid['seasonal_order'],
                            enforce_stationarity=False,
                            enforce_invertibility=False,
                            trend='c')
            result = model.fit(disp=False)
            forecast = result.forecast(steps=horizon, exog=exog_test)
            rmse = np.sqrt(mean_squared_error(y_test, forecast))
            rmse_list.append(rmse)
        except:
            rmse_list.append(np.inf)
    return np.mean(rmse_list)

print('Helpers defined.')

Helpers defined.


## Hyperparameter Tuning via Cross-Validation

We search the parameter grid using expanding-window cross-validation **on the training set only**.
For each section, we evaluate all combinations of (p,d,q) and (P,D,Q,7) and select the one that minimises RMSE averaged over the CV folds.
The `TimeSeriesSplit` uses 5 splits with test size = `FORECAST_HORIZON` (7 days).

In [ ]:
# Build full parameter list
param_combinations = []
for p in P_GRID:
    for d in D_GRID:
        for q in Q_GRID:
            for P in P_SEASONAL_GRID:
                for D in D_SEASONAL_GRID:
                    for Q in Q_SEASONAL_GRID:
                        param_combinations.append({
                            'order': (p,d,q),
                            'seasonal_order': (P,D,Q,SEASONAL_PERIOD)
                        })

print(f'{len(param_combinations)} parameter combinations to evaluate per section.')

best_params_per_section = {}

for sec in sections:
    print(f'\n{"="*55}\nTuning section {sec}\n{"="*55}')
    df_sec = daily[daily['section'] == sec].sort_values('ds').reset_index(drop=True)
    train_df, val_df, test_df = split_section(df_sec)

    if len(train_df) < 30:
        print(f'  Not enough training data ({len(train_df)} rows), using default (1,0,1)(0,0,0,7).')
        best_params_per_section[sec] = {'order': (1,0,1), 'seasonal_order': (0,0,0,SEASONAL_PERIOD)}
        continue

    # Prepare data for CV (use train set only)
    y_train = train_df['y']
    exog_train = train_df[['foot_traffic', 'is_holiday', 'is_special_day']]

    best_rmse = np.inf
    best_params = None

    for params in param_combinations:
        try:
            rmse = expand_window_cv(y_train, exog_train, params, horizon=FORECAST_HORIZON, n_splits=5)
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = params
        except Exception as e:
            pass

    if best_params is None:
        best_params = {'order': (1,0,1), 'seasonal_order': (0,0,0,SEASONAL_PERIOD)}
    best_params_per_section[sec] = best_params
    print(f'  Best CV RMSE: {best_rmse:.4f}')
    print(f'  Best params : order={best_params["order"]}, seasonal_order={best_params["seasonal_order"]}')

print('\nTuning complete.')

64 parameter combinations to evaluate per section.

Tuning section A


## Train Final Models & Evaluate on Test Set

Each section's model is retrained on **train + validation** data using the best hyperparameters,
then evaluated on the held‑out test set.

In [ ]:
all_metrics = []
trained_models = {}

for sec in sections:
    print(f'\n{"="*55}\nFinal training – section {sec}\n{"="*55}')
    df_sec = daily[daily['section'] == sec].sort_values('ds').reset_index(drop=True)
    train_df, val_df, test_df = split_section(df_sec)

    # Combine train + val for final fit
    trainval_df = pd.concat([train_df, val_df]).sort_values('ds').reset_index(drop=True)
    y_trainval = trainval_df['y']
    exog_trainval = trainval_df[['foot_traffic', 'is_holiday', 'is_special_day']]

    params = best_params_per_section[sec]
    model = SARIMAX(y_trainval, exog=exog_trainval,
                    order=params['order'],
                    seasonal_order=params['seasonal_order'],
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    trend='c')
    result = model.fit(disp=False)

    # Predict on test set
    exog_test = test_df[['foot_traffic', 'is_holiday', 'is_special_day']]
    forecast = result.forecast(steps=len(test_df), exog=exog_test)
    # Convert back from log scale
    y_true_orig = np.expm1(test_df['y'])
    y_pred_orig = np.expm1(forecast).clip(lower=0)

    m_dict = metrics(y_true_orig, y_pred_orig)
    print(f'  RMSE={m_dict["RMSE"]:.3f}  MAE={m_dict["MAE"]:.3f}  '
          f'MAPE={m_dict["MAPE"]:.1f}%  R2={m_dict["R2"]:.3f}')

    all_metrics.append({'Section': sec, **m_dict})
    trained_models[sec] = {
        'model': result,
        'test_df': test_df.assign(y_true=y_true_orig, y_pred=y_pred_orig),
        'trainval_df': trainval_df,
        'params': params
    }

print('\nAll sections trained.')

## Test‑Set Performance Summary

In [ ]:
summary_df = pd.DataFrame(all_metrics).set_index('Section')
summary_df = summary_df.round(4)
print(summary_df.to_string())
summary_df.to_csv(f'{MODEL_DIR}/test_metrics_SARIMA.csv')
print('\nSaved to deployment_models/test_metrics_SARIMA.csv')

## Predictions vs Actuals on Test Set

Each subplot shows the actual daily waste (blue) and the SARIMA forecast (orange).
No uncertainty intervals are shown as SARIMAX does not produce them by default (though they can be extracted from `result.get_forecast(...)`).

In [ ]:
fig, axes = plt.subplots(len(sections), 1, figsize=(14, 4 * len(sections)), sharex=False)
if len(sections) == 1:
    axes = [axes]

for ax, sec in zip(axes, sections):
    tdf = trained_models[sec]['test_df'].sort_values('ds')
    ax.plot(tdf['ds'], tdf['y_true'], color='steelblue', lw=1.5, marker='o', markersize=3, label='Actual')
    ax.plot(tdf['ds'], tdf['y_pred'], color='darkorange', lw=1.5, linestyle='--', marker='x', markersize=3, label='Forecast')
    ax.set_title(f'Section {sec} – Test Set Forecast vs Actual', fontsize=11)
    ax.set_ylabel('Waste (kg)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/SARIMA_test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## Weekly Forecast Function

Given a start date, predict the next 7 days of waste for every canteen section.
Regressor values for future dates are estimated from the most recent 7‑day average of foot traffic.

In [ ]:
def forecast_week(start_date: str, section: str = None):
    """
    Predict daily waste for the 7 days starting from `start_date`.

    Parameters
    ----------
    start_date : str  e.g. '2025-09-01'
    section    : str or None  – if None, forecasts all sections

    Returns
    -------
    pd.DataFrame with columns [ds, section, yhat, yhat_lower, yhat_upper]
    Note: For SARIMA, yhat_lower and yhat_upper are not provided; we set them to None.
    """
    start = pd.Timestamp(start_date)
    future_dates = pd.date_range(start, periods=FORECAST_HORIZON, freq='D')
    secs = [section] if section else sections
    results = []

    for sec in secs:
        info = trained_models[sec]
        tv   = info['trainval_df']

        # Estimate future foot traffic as the rolling 7‑day mean of the last week of training
        last_7_mean = tv.set_index('ds')['foot_traffic'].rolling(7).mean().iloc[-1]
        future_df = pd.DataFrame({
            'ds': future_dates,
            'foot_traffic': last_7_mean,
            'is_holiday': 0,
            'is_special_day': 0,
        })

        exog_future = future_df[['foot_traffic', 'is_holiday', 'is_special_day']]
        forecast = info['model'].forecast(steps=FORECAST_HORIZON, exog=exog_future)
        # Invert log transform
        forecast_orig = np.expm1(forecast).clip(lower=0)

        res_df = pd.DataFrame({
            'ds': future_dates,
            'section': sec,
            'yhat': forecast_orig,
            'yhat_lower': np.nan,
            'yhat_upper': np.nan
        })
        results.append(res_df)

    return pd.concat(results).reset_index(drop=True)

# Demo: forecast the week starting 2025-09-01
demo = forecast_week('2025-09-01')
print(demo.to_string(index=False))

### Visualise the 7‑Day Forecast

In [ ]:
demo_pivot = demo.pivot(index='ds', columns='section', values='yhat')
demo_pivot.plot(figsize=(10, 4), marker='o', title='7‑Day Waste Forecast per Section (demo)')
plt.ylabel('Predicted Waste (kg)')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Save Models

In [ ]:
for sec in sections:
    info = trained_models[sec]
    artifact = {
        'section':  sec,
        'model':    info['model'],
        'params':   info['params'],
        'train_end': str(train_end.date()),
        'val_end':   str(val_end.date()),
        'test_start': str(test_start.date()),
    }
    path = f'{MODEL_DIR}/section_{sec}_SARIMA.joblib'
    joblib.dump(artifact, path)
    print(f'Saved {path}')